# Chat Model Event Streams

The `chat_model_stream.py` module defines synchronous and asynchronous per-message streaming objects for version 3 chat-model event streams.

`ChatModelStream` is returned by `BaseChatModel.stream_events(version="v3")`, while `AsyncChatModelStream` is returned by `BaseChatModel.astream_events(version="v3")`.

Both stream variants consume content-block protocol events, accumulate message state, provide typed projections, preserve raw events for replay, support multiple independent consumers, and assemble a final `AIMessage` using version 1 content blocks. 

# Public Exports

The module explicitly exports the public stream and projection classes.

**Syntax**

```python
__all__ = [
    "AsyncChatModelStream",
    "AsyncProjection",
    "ChatModelStream",
    "SyncProjection",
    "SyncTextProjection",
]
```

# Shared Projection Behaviour

`SyncProjection`, `SyncTextProjection`, and `AsyncProjection` inherit producer-side lifecycle behaviour from the internal `_ProjectionBase` class.

The producer pushes deltas into a replay buffer and eventually completes the projection with a final value or fails it with an exception.

## Inherited Properties

1. `done`:`bool`:= Indicates whether the projection has completed successfully or failed.

2. `error`:`BaseException | None`:= Returns the terminal exception when the projection has failed.

## Inherited Methods

1. `push`:= Appends one delta to the projection's replay buffer.

   Existing consumers and consumers created later can observe the delta. This method is primarily used by the owning stream.

   **Syntax**

   ```python
   push(
       self, # Projection instance
       delta: Any # Delta value to append
   ) -> None
   ```

2. `complete`:= Stores the final accumulated value and marks the projection as successfully completed.

   **Syntax**

   ```python
   complete(
       self, # Projection instance
       final_value: Any # Final accumulated projection value
   ) -> None
   ```

3. `fail`:= Stores a terminal exception and marks the projection as finished.

   Consumers raise the stored exception when they next drain or iterate over the projection.

   **Syntax**

   ```python
   fail(
       self, # Projection instance
       error: BaseException # Terminal projection error
   ) -> None
   ```

# SyncProjection: `_ProjectionBase`

`SyncProjection` is a synchronous replayable iterable of delta values.

When a consumer reaches the end of the current buffer before completion, the projection can invoke a configured pull callback to request another source event. Every new iterator starts at the beginning of the replay buffer.

## Methods

1. `__init__`:= Creates an empty synchronous projection without a lazy-start or source-pump callback.

   **Syntax**

   ```python
   __init__(
       self # Synchronous projection instance
   ) -> None
   ```

2. `set_start`:= Installs a callback that runs before the projection is consumed for the first time.

   Passing `None` removes the callback.

   **Syntax**

   ```python
   set_start(
       self, # Synchronous projection instance
       cb: Callable[[], None] | None # Lazy-start callback
   ) -> None
   ```

3. `set_request_more`:= Installs the callback used to request another source event when a consumer reaches the end of the current buffer.

   The callback returns `True` when another event was produced and `False` when the source is exhausted. Passing `None` removes the callback.

   **Syntax**

   ```python
   set_request_more(
       self, # Synchronous projection instance
       cb: Callable[[], bool] | None # Source-pump callback
   ) -> None
   ```

4. `__iter__`:= Returns a synchronous iterator over buffered and subsequently produced deltas.

   Each iterator begins at the first buffered delta. When no pull callback is configured, iteration stops after the currently available values unless the projection is already complete.

   The stored terminal exception is raised when the projection has failed.

   **Syntax**

   ```python
   __iter__(
       self # Synchronous projection instance
   ) -> Iterator[Any]
   ```

5. `get`:= Drains the source through the pull callback and returns the final accumulated value.

   The lazy-start callback runs first when configured. The stored terminal exception is raised when the projection has failed.

   When the source is exhausted without calling `complete()`, the currently stored final value is returned and may be `None`.

   **Syntax**

   ```python
   get(
       self # Synchronous projection instance
   ) -> Any
   ```

# SyncTextProjection: `SyncProjection`

`SyncTextProjection` specialises `SyncProjection` for text and reasoning content.

It retains normal delta iteration while adding string conversion, truth-value testing, and a text-oriented representation.

## Methods

1. `__str__`:= Drains the projection and returns the complete accumulated string.

   An unset or `None` final value is represented as an empty string.

   **Syntax**

   ```python
   __str__(
       self # Text projection instance
   ) -> str
   ```

2. `__bool__`:= Returns whether at least one delta has been pushed.

   This check does not drain the projection.

   **Syntax**

   ```python
   __bool__(
       self # Text projection instance
   ) -> bool
   ```

3. `__repr__`:= Returns a representation of the final value when the projection is complete.

   Before completion, it returns a representation of the currently concatenated delta strings without requesting additional events.

   **Syntax**

   ```python
   __repr__(
       self # Text projection instance
   ) -> str
   ```

# AsyncProjection: `_ProjectionBase`

`AsyncProjection` is an asynchronous replayable iterable that can also be awaited for its final accumulated value.

An `asyncio.Event` notifies consumers whenever the projection state changes. Producers and consumers must use the same event loop. Every asynchronous iterator has an independent cursor beginning at the first buffered delta.

## Methods

1. `__init__`:= Creates an empty asynchronous projection without a lazy-start or source-pump callback.

   **Syntax**

   ```python
   __init__(
       self # Asynchronous projection instance
   ) -> None
   ```

2. `set_start`:= Installs an asynchronous callback that runs before the projection is consumed for the first time.

   Passing `None` removes the callback.

   **Syntax**

   ```python
   set_start(
       self, # Asynchronous projection instance
       cb: Callable[[], Awaitable[None]] | None # Asynchronous lazy-start callback
   ) -> None
   ```

3. `set_arequest_more`:= Installs the asynchronous callback used to request another source event when a consumer reaches the end of the current buffer.

   The callback returns `True` when another event was produced and `False` when the source is exhausted. Passing `None` removes the callback.

   **Syntax**

   ```python
   set_arequest_more(
       self, # Asynchronous projection instance
       cb: Callable[[], Awaitable[bool]] | None # Asynchronous source-pump callback
   ) -> None
   ```

4. `push`:= Appends one delta to the replay buffer and wakes waiting consumers.

   **Syntax**

   ```python
   push(
       self, # Asynchronous projection instance
       delta: Any # Delta value to append
   ) -> None
   ```

5. `complete`:= Stores the final accumulated value, marks the projection complete, and wakes waiting consumers.

   **Syntax**

   ```python
   complete(
       self, # Asynchronous projection instance
       final_value: Any # Final accumulated projection value
   ) -> None
   ```

6. `fail`:= Stores a terminal exception, marks the projection finished, and wakes waiting consumers.

   **Syntax**

   ```python
   fail(
       self, # Asynchronous projection instance
       error: BaseException # Terminal projection error
   ) -> None
   ```

7. `__aiter__`:= Returns a new asynchronous iterator over projection deltas.

   Each iterator replays the complete buffer from the beginning and then waits for or requests additional values.

   **Syntax**

   ```python
   __aiter__(
       self # Asynchronous projection instance
   ) -> AsyncIterator[Any]
   ```

8. `__await__`:= Makes the projection awaitable for its final accumulated value.

   When a pull callback is configured, awaiting the projection drives that callback until completion or source exhaustion. Otherwise, it waits for producer notifications.

   The stored terminal exception is raised when the projection has failed.

   **Syntax**

   ```python
   __await__(
       self # Asynchronous projection instance
   ) -> Generator[Any, None, Any]
   ```

# Shared Chat Model Stream Behaviour

`ChatModelStream` and `AsyncChatModelStream` inherit event accumulation and metadata behaviour from the internal `_ChatModelStreamBase` class.

The shared implementation processes message and content-block protocol events, accumulates typed values, finalises tool-call chunks, and assembles the final `AIMessage`.

## Inherited Properties

1. `namespace`:`list[str]`:= Returns the graph namespace path associated with the streamed message.

2. `node`:`str | None`:= Returns the graph node that produced the message.

3. `message_id`:`str | None`:= Returns the stable message identifier when one is available.

4. `done`:`bool`:= Indicates whether the stream has completed or failed.

5. `has_events`:`bool`:= Indicates whether at least one raw protocol event has been recorded.

6. `output_message`:`AIMessage | None`:= Returns the assembled message after successful stream completion.

   This property does not start the source, request events, block, or raise the stored stream error. It returns `None` while no output message is available.

## Inherited Methods

1. `set_message_id`:= Assigns the stable message identifier after the underlying chat-model run starts.

   This method is intended for the stream driver rather than normal consumer code.

   **Syntax**

   ```python
   set_message_id(
       self, # Chat-model stream instance
       message_id: str # Stable message identifier
   ) -> None
   ```

2. `dispatch`:= Processes one content-block protocol event.

   Supported event types include `"message-start"`, `"content-block-delta"`, `"content-block-finish"`, `"message-finish"`, and `"error"`.

   Every event is first appended to the raw replay buffer. A `"content-block-start"` event is retained in the buffer but requires no accumulation work.

   An `"error"` event fails the stream with a `RuntimeError`.

   **Syntax**

   ```python
   dispatch(
       self, # Chat-model stream instance
       event: Mapping[str, Any] # Content-block protocol event
   ) -> None
   ```

3. `fail`:= Marks the stream as failed and propagates the exception to its projections.

   The asynchronous stream also fails its output and raw-event projections.

   **Syntax**

   ```python
   fail(
       self, # Chat-model stream instance
       error: BaseException # Terminal stream error
   ) -> None
   ```

## Event Accumulation

The shared implementation performs the following processing:

* Text deltas are accumulated globally and by content-block index.
* Reasoning deltas are accumulated globally and by content-block index.
* Tool-call chunks retain their first non-empty identifier and name while argument fragments are accumulated.
* Server-side tool-call chunks are accumulated separately and do not appear in the public `tool_calls` projection.
* Tool-call argument strings are parsed when their blocks or the message finish.
* Unparseable tool calls become invalid tool calls.
* Finished blocks are retained by event index so the final content order matches the protocol order.
* Message-start metadata may provide the model provider, model name, and message identifier.
* Message-finish data may provide usage metadata, response metadata, and provider-specific additional keyword arguments.

## Final Output Assembly

The final `AIMessage` contains:

* Ordered finalised content blocks.
* The stable message identifier.
* Finalised client-side tool calls.
* Invalid tool calls.
* Usage metadata.
* Response metadata.
* Optional provider-specific additional keyword arguments.

When protocol blocks were received, `AIMessage.content` is a list of version 1 content-block dictionaries. When no protocol blocks were received, the accumulated text is used as string content.

The output metadata always sets the output version to `"v1"`.

**Syntax**

```python
response_metadata["output_version"] = "v1"
```

This prevents already normalised version 1 blocks from being translated again by another output-version handler.

# ChatModelStream: `_ChatModelStreamBase`

`ChatModelStream` is the synchronous per-message stream returned by `BaseChatModel.stream_events(version="v3")`.

It exposes synchronous projections, replayable raw-event iteration, and a blocking `output` property.

**Syntax**

```python
ChatModelStream(
    *,
    namespace: list[str] | None = None, # Graph namespace path
    node: str | None = None, # Graph node producing the message
    message_id: str | None = None # Initial stable message identifier
)
```

## Properties

1. `text`:`SyncTextProjection`:= Returns the cached text projection.

   Iterating over it yields text deltas. Converting it with `str()` drains the stream and returns the complete text.

2. `reasoning`:`SyncTextProjection`:= Returns the cached reasoning projection.

   It has the same behaviour as `text`.

3. `tool_calls`:`SyncProjection`:= Returns the cached tool-call projection.

   Iteration yields `ToolCallChunk` deltas. Calling `get()` returns the finalised `list[ToolCall]`.

4. `output`:`AIMessage`:= Drains the remaining source events and returns the assembled message.

   The stored stream exception is raised when the stream has failed. A `RuntimeError` is raised when the source finishes without producing a message.

## Methods

1. `__init__`:= Creates an empty synchronous chat-model stream.

   Text and reasoning projections are created as `SyncTextProjection` objects. Tool calls use a `SyncProjection`.

   **Syntax**

   ```python
   __init__(
       self, # Synchronous chat-model stream instance
       *,
       namespace: list[str] | None = None, # Graph namespace path
       node: str | None = None, # Graph node producing the message
       message_id: str | None = None # Initial stable message identifier
   ) -> None
   ```

2. `bind_pump`:= Binds a standalone source-pump callback.

   This method delegates to `set_request_more()` and is used by synchronous version 3 chat-model event streaming.

   **Syntax**

   ```python
   bind_pump(
       self, # Synchronous chat-model stream instance
       pump_one: Callable[[], bool] # Callback that requests one additional source event
   ) -> None
   ```

3. `set_start`:= Installs a lazy-start callback on the stream and all synchronous projections.

   Passing `None` removes the callback.

   **Syntax**

   ```python
   set_start(
       self, # Synchronous chat-model stream instance
       cb: Callable[[], None] | None # Lazy-start callback
   ) -> None
   ```

4. `set_request_more`:= Installs one source-pump callback on the stream and all synchronous projections.

   The callback returns `True` when another event was produced and `False` when the source is exhausted.

   **Syntax**

   ```python
   set_request_more(
       self, # Synchronous chat-model stream instance
       cb: Callable[[], bool] # Shared source-pump callback
   ) -> None
   ```

5. `__iter__`:= Returns a synchronous iterator over raw protocol events.

   Every iterator starts at the beginning of the replay buffer. When it reaches the end of the available events, it requests additional events through the configured pump until the stream completes or the source is exhausted.

   The stored terminal exception is raised when the stream has failed.

   **Syntax**

   ```python
   __iter__(
       self # Synchronous chat-model stream instance
   ) -> Iterator[MessagesData]
   ```

# AsyncChatModelStream: `_ChatModelStreamBase`

`AsyncChatModelStream` is the asynchronous per-message stream returned by `BaseChatModel.astream_events(version="v3")`.

It can be awaited for the final `AIMessage` and asynchronously iterated over for raw protocol events. Its projections are also asynchronous iterables and awaitable objects.

**Syntax**

```python
AsyncChatModelStream(
    *,
    namespace: list[str] | None = None, # Graph namespace path
    node: str | None = None, # Graph node producing the message
    message_id: str | None = None # Initial stable message identifier
)
```

## Properties

1. `text`:`AsyncProjection`:= Returns the text projection.

   Asynchronous iteration yields text deltas. Awaiting the projection returns the complete text.

2. `reasoning`:`AsyncProjection`:= Returns the reasoning projection.

   It has the same asynchronous iterable and awaitable behaviour as `text`.

3. `tool_calls`:`AsyncProjection`:= Returns the tool-call projection.

   Asynchronous iteration yields `ToolCallChunk` deltas. Awaiting it returns the finalised tool-call list.

4. `output`:`AsyncProjection`:= Returns the output projection.

   Awaiting it returns the assembled `AIMessage`. Awaiting the complete stream also waits for the producer task and post-stream callbacks.

## Methods

1. `__init__`:= Creates an empty asynchronous chat-model stream.

   Text, reasoning, tool-call, output, and raw-event projections are created as `AsyncProjection` objects.

   **Syntax**

   ```python
   __init__(
       self, # Asynchronous chat-model stream instance
       *,
       namespace: list[str] | None = None, # Graph namespace path
       node: str | None = None, # Graph node producing the message
       message_id: str | None = None # Initial stable message identifier
   ) -> None
   ```

2. `set_arequest_more`:= Installs one asynchronous source-pump callback on every projection.

   The callback returns `True` when another source event was produced and `False` when the source is exhausted. Passing `None` removes the callback.

   **Syntax**

   ```python
   set_arequest_more(
       self, # Asynchronous chat-model stream instance
       cb: Callable[[], Awaitable[bool]] | None # Shared asynchronous source-pump callback
   ) -> None
   ```

3. `set_start`:= Installs one asynchronous lazy-start callback on the stream and every projection.

   Passing `None` removes the callback.

   **Syntax**

   ```python
   set_start(
       self, # Asynchronous chat-model stream instance
       cb: Callable[[], Awaitable[None]] | None # Asynchronous lazy-start callback
   ) -> None
   ```

4. `__await__`:= Makes the stream awaitable for the assembled `AIMessage`.

   After the output projection resolves, the method also awaits the producer task so post-stream work, including completion callbacks, finishes before the message is returned.

   **Syntax**

   ```python
   __await__(
       self # Asynchronous chat-model stream instance
   ) -> Generator[Any, None, AIMessage]
   ```

5. `__aiter__`:= Returns an asynchronous iterator over raw protocol events.

   The iterator uses the raw-event replay projection, so every consumer starts at the first retained event.

   **Syntax**

   ```python
   __aiter__(
       self # Asynchronous chat-model stream instance
   ) -> AsyncIterator[MessagesData]
   ```

6. `aclose`:= Cancels or awaits the background producer and releases stream resources.

   When no successful output has been produced, an active producer task is cancelled and the stream fails with `asyncio.CancelledError`.

   When output has been produced but post-stream work is still running, the task is awaited instead of cancelled so tracing completion callbacks are preserved.

   This method is idempotent and may be called before, during, or after normal completion.

   **Syntax**

   ```python
   async aclose(
       self # Asynchronous chat-model stream instance
   ) -> None
   ```

7. `__aenter__`:= Enters the asynchronous context manager and returns the stream.

   **Syntax**

   ```python
   async __aenter__(
       self # Asynchronous chat-model stream instance
   ) -> Self
   ```

8. `__aexit__`:= Exits the asynchronous context manager by calling `aclose()`.

   **Syntax**

   ```python
   async __aexit__(
       self, # Asynchronous chat-model stream instance
       exc_type: type[BaseException] | None, # Exception type from the context
       exc: BaseException | None, # Exception raised inside the context
       tb: object # Traceback object
   ) -> None
   ```
